In [51]:
import pandas as pd

In [52]:
df = pd.read_csv("IMDB Dataset.csv")

In [53]:
df.shape

(50000, 2)

In [54]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [55]:
df.drop_duplicates(inplace=True)

In [56]:
df.shape

(49582, 2)

# Pre Processing

### 1. Coverting test to lowercase

In [57]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [58]:
import re

def remove_urls(text):
    text = re.sub(r"http\S+","",text)
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Removing punctuations

In [59]:
def remove_punctuations(text):
    text = re.sub(r"^[A-Za-z0-9\s+]","",text)
    return text
df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing HTML

In [60]:
def remove_HTML(text):
    text = re.sub(r"<.*?>","",text)
    return text
df["review"] = df["review"].apply(remove_HTML)

### 5. Removing the Stopwords

In [61]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\Sagar
[nltk_data]     Panwar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Sagar
[nltk_data]     Panwar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Sagar
[nltk_data]     Panwar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [62]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")

    return text
df["review"] = df["review"].apply(remove_stopwords)

### 6. Stemming

In [63]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
        
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

### 7. Encoding

In [64]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

### 8. Vectorization

In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])
y = df["sentiment"]

## Dataset and DataLoader

In [67]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [73]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [74]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [79]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

C:\Users\Sagar Panwar\AppData\Local\Temp\ipykernel_4292\2931448922.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(y_train.values).float()


In [80]:
train_loader = DataLoader(train_set,batch_size=64,shuffle=True)
test_loader = DataLoader(test_set,batch_size=64,shuffle=True)

### Build RNN

In [1]:
import torch.nn as nn
import torch.optim as optim

In [3]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        ## RNN layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layers, batch_first=True)
        
        # fully connected layer
        self.fc = nn.Linear(hidden_size,1)

    def forward(self,x):
        #optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_ = self.rnn(x,h0)
        # 1st value = hidden state of all the timesteps
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:,-1,:])
        return out
        
        